<a href="https://colab.research.google.com/github/jpnoug/spectropy/blob/main/selection_cibles_etoiles_spectro_Simbad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sélection de cibles pour spectroscopie stellaire

Ce notebook interroge la base de données **SIMBAD** via `astroquery` pour lister des étoiles candidates selon :
- une **constellation** donnée
- une plage de **magnitude V**
- un ou plusieurs **types spectraux** (O B A F G K M, classes de luminosité I–V)

Les résultats sont filtrés, présentés en tableaux et exportés en csv.

In [1]:
# ── Installation des dépendances ──────────────────────────────────
!pip install astroquery astropy pandas plotly ipywidgets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.0 MB/s eta 0:00:00


In [2]:
# ── Imports ───────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import re
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
import astropy.units as u

print('✅ Imports OK')

✅ Imports OK


---
## ① Paramètres de la requête
Modifier les valeurs ci-dessous avant d'exécuter les cellules suivantes.

In [8]:
from inspect import ClassFoundException

# ── Paramètres utilisateur ───────────────────────────────────────

# Constellation (abréviation IAU 3 lettres)
# Exemples : 'Ori', 'Cyg', 'Sco', 'Lyr', 'Tau', 'Leo', 'CMa', 'Vir', 'Aql'
CONSTELLATION = 'Her'

# Plage de magnitude V
MAG_MIN = 2.0
MAG_MAX = 8.0

# Filtre de type spectral — préfixes MK acceptés (liste)
# Exemples :
#   ['A']          → tous les A (A0, A1, A2 ... A9)
#   ['A1']         → uniquement A1x (A1V, A1Ia, A1III...)
#   ['A1', 'A2']   → A1 et A2
#   ['B', 'A0']    → tous les B + les A0
#   []             → tous les types spectraux
FILTRE_SPECTRAL = ['A']

# Classes de luminosité à conserver (liste)
# Laisser vide [] pour toutes les classes
# CLASSES_LUMINOSITE = ['I', 'II', 'III', 'IV', 'V']
CLASSES_LUMINOSITE = ['V']

# ── Rappel  ───────────────────────────────────────────────────────────
print('Paramètres :')
print(f'  Constellation    : {CONSTELLATION}')
print(f'  Magnitude V      : {MAG_MIN} ≤ V < {MAG_MAX}')
print(f'  Filtre spectral  : {FILTRE_SPECTRAL if FILTRE_SPECTRAL else "tous"}')
print(f'  Classes lum.     : {CLASSES_LUMINOSITE if CLASSES_LUMINOSITE else "toutes"}')


Paramètres :
  Constellation    : Her
  Magnitude V      : 2.0 ≤ V < 8.0
  Filtre spectral  : ['A']
  Classes lum.     : ['V']


---
## ② Requête SIMBAD
On utilise le **TAP service SIMBAD** (`Simbad.query_tap`) avec une requête ADQL, plus flexible et complète que l'API REST classique.

In [9]:
# ── Cellule 5 : Requête SIMBAD via TAP/ADQL (filtrage côté serveur) ──────────

CONST_CENTERS = {
    'And': (11.1, 37.7),  'Ant': (150.0, -30.0), 'Aps': (250.0, -75.0),
    'Aql': (297.0,  3.0), 'Aqr': (339.0, -11.0), 'Ara': (267.0, -57.0),
    'Ari': ( 28.7, 20.8), 'Aur': ( 90.0,  42.0), 'Boo': (213.0,  31.0),
    'CMa': (106.0,-22.0), 'CMi': (115.0,   6.0), 'CVn': (192.0,  40.0),
    'Cap': (315.0,-18.0), 'Car': (130.0, -60.0), 'Cas': ( 12.0,  62.0),
    'Cen': (182.0,-47.0), 'Cep': (322.0,  71.0), 'Cet': ( 26.0,  -7.0),
    'Cir': (230.0,-63.0), 'CrA': (279.0, -41.0), 'CrB': (233.0,  33.0),
    'Crv': (187.0,-18.0), 'Crt': (173.0, -14.0), 'Cru': (187.0, -60.0),
    'Cyg': (311.0, 44.0), 'Del': (309.0,  13.0), 'Dor': ( 82.0, -59.0),
    'Dra': (240.0, 67.0), 'Equ': (321.0,   7.0), 'Eri': ( 55.0, -28.0),
    'For': ( 46.0,-30.0), 'Gem': (103.0,  22.0), 'Gru': (333.0, -46.0),
    'Her': (254.0, 27.0), 'Hor': ( 45.0, -54.0), 'Hya': (140.0, -14.0),
    'Hyi': ( 42.0,-70.0), 'Ind': (321.0, -59.0), 'LMi': (162.0,  32.0),
    'Lac': (337.0, 46.0), 'Leo': (168.0,  13.0), 'Lep': ( 84.0, -19.0),
    'Lib': (229.0,-15.0), 'Lup': (234.0, -43.0), 'Lyn': (135.0,  47.0),
    'Lyr': (283.0, 36.0), 'Men': ( 84.0, -77.0), 'Mic': (311.0, -36.0),
    'Mon': (112.0,  2.0), 'Mus': (193.0, -70.0), 'Nor': (243.0, -51.0),
    'Oct': (270.0,-85.0), 'Oph': (257.0,  -8.0), 'Ori': ( 83.0,   5.0),
    'Pav': (283.0,-66.0), 'Peg': (340.0,  20.0), 'Per': ( 52.0,  45.0),
    'Phe': ( 16.0,-49.0), 'Pic': ( 90.0, -53.0), 'PsA': (332.0, -32.0),
    'Psc': (  8.0, 14.0), 'Pup': (120.0, -31.0), 'Pyx': (136.0, -27.0),
    'Ret': ( 63.0,-63.0), 'Scl': ( 12.0, -33.0), 'Sco': (253.0, -28.0),
    'Sct': (281.0, -9.0), 'Ser': (237.0,   8.0), 'Sex': (153.0,  -2.0),
    'Sge': (298.0, 19.0), 'Sgr': (283.0, -27.0), 'Tau': ( 67.0,  19.0),
    'Tel': (290.0,-51.0), 'TrA': (248.0, -65.0), 'Tri': ( 32.0,  31.0),
    'Tuc': (354.0,-65.0), 'UMa': (165.0,  55.0), 'UMi': (217.0,  78.0),
    'Vel': (142.0,-47.0), 'Vir': (190.0,  -4.0), 'Vol': (122.0, -69.0),
    'Vul': (298.0, 24.0),
}

if CONSTELLATION not in CONST_CENTERS:
    raise ValueError(
        f"Constellation '{CONSTELLATION}' inconnue. "
        "Utilisez une abréviation IAU 3 lettres, ex : 'Ori', 'Cyg', 'Sco'."
    )

ra_c, dec_c = CONST_CENTERS[CONSTELLATION]
rayon_deg = 15.0

# Filtre ADQL côté serveur : seulement sur la lettre MK (premier caractère)
# Le filtre sur le sous-type (ex: '1' dans 'A1') est fait en Python (cellule 6)
# pour gérer les variantes SIMBAD : 'A1V', 'A1.5V', 'A1-2V', 'A1/A2'...
if FILTRE_SPECTRAL:
    mk_uniques = list({p[0].upper() for p in FILTRE_SPECTRAL if p})
    sp_likes = " OR ".join(f"sp_type LIKE '{mk}%'" for mk in sorted(mk_uniques))
    sp_clause = f"AND ({sp_likes})"
else:
    sp_clause = "AND sp_type IS NOT NULL"

adql = f"""SELECT TOP 1000
    main_id, ra, dec, sp_type, plx_value AS parallax_mas, flux AS mag_V
FROM flux
JOIN basic ON oidref = oid
WHERE filter = 'V'
AND flux BETWEEN {MAG_MIN} AND {MAG_MAX}
AND sp_type IS NOT NULL
{sp_clause}
AND CONTAINS(POINT('ICRS', ra, dec), BOX('ICRS', {ra_c}, {dec_c}, {rayon_deg*2}, {rayon_deg*2})) = 1
ORDER BY mag_V ASC"""

print(f"Requête SIMBAD — {CONSTELLATION} | V=[{MAG_MIN}, {MAG_MAX}] | filtre={FILTRE_SPECTRAL or 'tous'}")
print(f"Zone : BOX({ra_c}°, {dec_c}°) ± {rayon_deg}°")
result_table = Simbad.query_tap(adql)

n = len(result_table) if result_table is not None else 0
print(f"\n✅ {n} objets retournés par SIMBAD (avant filtrage sous-type et classe).")


Requête SIMBAD — Her | V=[2.0, 8.0] | filtre=['A']
Zone : BOX(254.0°, 27.0°) ± 15.0°

✅ 148 objets retournés par SIMBAD (avant filtrage sous-type et classe).


---
## ③ Filtrage et nettoyage des résultats

In [10]:
# ── Cellule 6 : Filtrage Python ───────────────────────────────────────────────

import re
import numpy as np
import pandas as pd

def parse_sptype(sp):
    """
    Extrait le type MK (lettre) et la classe de luminosité depuis une chaîne SIMBAD.
    'B2Iae' → ('B', '2', 'I') | 'A1V' → ('A', '1', 'V') | 'K5III' → ('K', '5', 'III')
    Retourne (mk, sous_type, lum_class).
    """
    if not sp or sp.strip() in ('', '~', '--'):
        return None, None, None
    sp = sp.strip()
    mk_match = re.match(r'^([OBAFGKM])', sp, re.IGNORECASE)
    if not mk_match:
        return None, None, None
    mk = mk_match.group(1).upper()

    # Sous-type numérique (0-9, éventuellement décimal)
    st_match = re.match(r'^[OBAFGKM]([0-9](?:\.[0-9])?)', sp, re.IGNORECASE)
    sous_type = st_match.group(1) if st_match else ''

    # Classe de luminosité — ordre d'importance décroissante
    lc_patterns = [
        (r'Iab', 'I'), (r'Ia', 'I'), (r'Ib', 'I'),
        (r'III', 'III'), (r'II', 'II'), (r'IV', 'IV'),
        (r'VI', None),
        (r'I(?![IVa-z0-9])', 'I'),
        (r'V(?![I])', 'V'),
    ]
    lum_class = None
    for pattern, lc_norm in lc_patterns:
        if re.search(pattern, sp):
            lum_class = lc_norm
            break

    return mk, sous_type, lum_class


def lc_to_label(lc):
    mapping = {
        'I': 'Supergéante', 'II': 'Géante brillante',
        'III': 'Géante', 'IV': 'Sous-géante', 'V': 'Naine'
    }
    return mapping.get(lc, lc or '?')


def match_filtre_spectral(mk, sous_type, filtre_spectral):
    """
    Vérifie si l'étoile correspond à un des préfixes du filtre.
    filtre_spectral = ['A1', 'B2', 'K'] par exemple.
    - 'A1' matche A1, A1V, A1Ia, A1.5V...
    - 'A'  matche tous les A
    """
    if not filtre_spectral:
        return True   # pas de filtre → tout passe
    if mk is None:
        return False
    for prefixe in filtre_spectral:
        prefixe = prefixe.strip().upper()
        if len(prefixe) == 1:
            # Filtre sur la lettre seule → accepte tous les sous-types
            if mk == prefixe:
                return True
        else:
            # Filtre sur lettre + chiffre(s) : ex 'A1'
            mk_filtre = prefixe[0]
            st_filtre = prefixe[1:]
            if mk == mk_filtre and sous_type.startswith(st_filtre):
                return True
    return False


# Conversion en DataFrame
df = result_table.to_pandas()
df.columns = [c.lower() for c in df.columns]
df = df.rename(columns={'mag_v': 'mag_V', 'mag_b': 'mag_B'})

# ── Récupération de mag_B ─────────────────────────────────────────────────────
ids_quoted = ', '.join(f"\'{i}\'" for i in df['main_id'].tolist())
adql_b = f"""SELECT main_id, flux AS mag_B
FROM flux
JOIN basic ON oidref = oid
WHERE filter = \'B\'
AND main_id IN ({ids_quoted})"""
try:
    res_b = Simbad.query_tap(adql_b)
    if res_b is not None and len(res_b) > 0:
        df_b = res_b.to_pandas()
        df_b.columns = [c.lower() for c in df_b.columns]
        df_b = df_b.rename(columns={'mag_b': 'mag_B'})
        df = df.merge(df_b[['main_id', 'mag_B']], on='main_id', how='left')
    else:
        df['mag_B'] = np.nan
except Exception as e:
    print(f'   Avertissement mag_B : {e}')
    df['mag_B'] = np.nan

# Calcul B-V
df['B_V'] = np.where(
    df['mag_B'].notna() & df['mag_V'].notna(),
    df['mag_B'] - df['mag_V'], np.nan
)

# Parsing type spectral
parsed = df['sp_type'].apply(parse_sptype)
df['mk_type']   = [p[0] for p in parsed]
df['sous_type'] = [p[1] for p in parsed]
df['lum_class'] = [p[2] for p in parsed]
df['lum_label'] = df['lum_class'].apply(lc_to_label)
df['sp_mk_st']  = df['mk_type'].fillna('?') + df['sous_type'].fillna('')  # ex: 'A1'

# ── Diagnostic avant filtrage ─────────────────────────────────────────────────
print(f'Objets reçus de SIMBAD : {len(df)}')
print('Distribution des sous-types spectraux :')
print(df['sp_mk_st'].value_counts().head(20).to_string())
print()
print('Distribution des classes de luminosité :')
print(df['lum_class'].value_counts(dropna=False).to_string())
print()

# ── Filtres ───────────────────────────────────────────────────────────────────
# Filtre spectral (préfixes)
mask_sp = df.apply(
    lambda r: match_filtre_spectral(r['mk_type'], r['sous_type'], FILTRE_SPECTRAL),
    axis=1
)
df = df[mask_sp]

# Filtre classe de luminosité
if CLASSES_LUMINOSITE:
    df = df[df['lum_class'].isin(CLASSES_LUMINOSITE)]

# Filtre magnitude V (redondant avec SIMBAD, mais sécurité)
df = df[(df['mag_V'] >= MAG_MIN) & (df['mag_V'] < MAG_MAX)]

# Tri par magnitude
df = df.sort_values('mag_V').reset_index(drop=True)

# Colonnes finales
cols_keep = ['main_id', 'ra', 'dec', 'mag_V', 'B_V',
             'sp_type', 'mk_type', 'sous_type', 'lum_class', 'lum_label']
if 'parallax_mas' in df.columns:
    cols_keep.append('parallax_mas')
df = df[[c for c in cols_keep if c in df.columns]]

print(f'✅ {len(df)} cibles après filtrage.')
if len(df) == 0:
    print('   → Aucune cible. Vérifiez FILTRE_SPECTRAL, CLASSES_LUMINOSITE, et la plage de magnitude.')
else:
    display(df)


Objets reçus de SIMBAD : 148
Distribution des sous-types spectraux :
sp_mk_st
A0    58
A2    31
A5    19
A3    18
A1    10
A8     3
A9     2
A7     2
A6     2
A4     2
A      1

Distribution des classes de luminosité :
lum_class
None    104
V        31
IV        7
III       4
I         2

✅ 31 cibles après filtrage.


,main_id,ra,dec,mag_V,B_V,sp_type,mk_type,sous_type,lum_class,lum_label,parallax_mas
0,* eps Her,255.072389,30.926404,3.920,-0.010,A0V,A,0,V,Naine,19.7639
1,* ome Her,246.353978,14.033267,4.580,-0.010,A2VpCrSr,A,2,V,Naine,13.1034
2,* e Her,259.417722,37.291506,4.650,0.050,A2V,A,2,V,Naine,18.6546
3,* pi. Ser,240.573715,22.804453,4.817,0.070,A3V,A,3,V,Naine,18.6950
4,* 60 Her,256.344543,12.740826,4.871,0.130,A3V,A,3,V,Naine,23.6844
5,* 70 Her,260.225858,24.499436,5.120,-0.030,A2V,A,2,V,Naine,7.4167
6,* rho Her B,260.919654,37.146785,5.398,-0.003,A0Vn,A,0,V,Naine,8.9108
7,* 25 Her,246.350703,37.394052,5.540,0.170,A5V,A,5,V,Naine,13.1903
8,HD 161833,266.783525,17.697023,5.610,0.030,A1V,A,1,V,Naine,7.6358
9,* 78 Her,262.956580,28.407501,5.650,0.000,A1V,A,1,V,Naine,11.9929


---
## ④ Décodage des qualificateurs spectraux


In [11]:
# ── Décodage des qualificateurs spectraux + tableau résultat ─────

import re
from IPython.display import display, HTML

# ── Dictionnaire des qualificateurs MK ───────────────────────────────────────
# Ordre : du plus spécifique au plus général (évite les sous-chaînes parasites)
QUALIFICATEURS = [
    # Combinaisons Ap multi-éléments (à tester EN PREMIER)
    (r'SrCrEu',                        'SrCrEu : Ap classique — Sr, Cr, Eu anormaux'),
    (r'SrEu',                          'SrEu : Ap-SrCrEu — Strontium + Europium'),
    (r'SiSr',                          'SiSr : Ap-SiSr — Silicium + Strontium'),
    (r'HgMn',                          'HgMn : CP — Mercure et Manganèse anormaux, pas de champ B'),
    # Éléments individuels
    (r'\bSi\b|\(Si\)|Si(?=[0-9])',     'Si : surabondance Silicium (Ap-Si, champ magnétique fort)'),
    (r'\bSr\b',                        'Sr : surabondance Strontium'),
    (r'\bEu\b',                        'Eu : surabondance Europium (Ap-SrCrEu)'),
    (r'\bCr\b',                        'Cr : surabondance Chrome'),
    (r'\bHg\b',                        'Hg : surabondance Mercure (HgMn)'),
    (r'\bMn\b',                        'Mn : surabondance Manganèse (HgMn)'),
    (r'\bBa\b',                        'Ba : étoile Barium — géante G/K enrichie en s-process'),
    # Hélium anormal (avant le pattern 'e' d'émission)
    (r'He-[wrs]',                      'He : Hélium anormal (He-weak ou He-strong/rich)'),
    # Étoiles Am / Ap
    (r'kA',                            'kA : Ca II de type A précoce — signature étoile Am'),
    (r'(?<![A-Z])Am(?![a-z])',         'Am : étoile métallique (Ca II faible, métaux forts)'),
    (r'(?<![a-z])Ap(?![a-z])',         'Ap : A particulière — champ magnétique, abondances anormales'),
    (r'(?<![a-z])Bp(?![a-z])',         'Bp : B particulière'),
    (r'(?<![A-Z])m(?=[A-Z]|$)',        'm : raies métalliques anormales (notation Am condensée)'),
    # Étoiles S, C, CH
    (r'\bCH\b',                        'CH : étoile CH — carbone et s-process, souvent binaire'),
    (r'\bS\b',                         'S : étoile S — bandes ZrO, enrichissement s-process'),
    # Émission et phénomènes de disque
    (r'Ia\+',                          'Ia+ : hypergéante — luminosité extrême, pertes de masse'),
    (r'shell|sh\b',                    'shell : étoile coquille — disque équatorial vu par la tranche'),
    (r'Be\b',                          'Be : étoile Be — Hα en émission, disque équatorial'),
    (r'Ae\b',                          'Ae : étoile Ae — Herbig Ae/Be ou émission'),
    (r'pe\b',                          'pe : particulière + émission'),
    # 'e' d'émission : exclure He- (déjà traité) et les fins de mots normaux
    (r'(?<!H)(?<!-)e(?=$|[^a-zA-Z])', 'e : émission dans les raies H (Hα, Hβ...)'),
    # Rotation : nn avant n (évite le double déclenchement)
    (r'nn\b',                          'nn : rotation très rapide — raies très élargies'),
    # n de rotation : exclure HgMn (précédé de M ou H), et ne (=émission)
    (r'(?<![HMhm])n(?!e)(?=$|[^a-zA-Z])', 'n : rotation rapide — raies élargies par v·sin i élevé'),
    (r'ss\b',                          'ss : rotation très lente — raies très fines'),
    (r's(?=$|[^a-zA-Z])',              's : rotation lente (sharp lines)'),
    # Qualité de classification
    (r':',                             ': : classification incertaine'),
    (r'\?',                            '? : type spectral douteux'),
    (r'(?<![Ia])\/(?!Ib)',             '/ : type intermédiaire entre deux classes'),
    (r'\+',                            '+ : binaire spectroscopique probable'),
    # Objets dégénérés
    (r'sdB\b',                         'sdB : sous-naine B chaude (EHB — Extreme Horizontal Branch)'),
    (r'sdO\b',                         'sdO : sous-naine O (post-HB)'),
    (r'\bDA\b',                        'DA : naine blanche H — raies Balmer seules'),
    (r'\bDB\b',                        'DB : naine blanche He'),
]

def decoder_sptype(sp):
    if not sp or sp.strip() in ('', '~', '--'):
        return ''
    notes, seen = [], set()
    for pattern, description in QUALIFICATEURS:
        if re.search(pattern, sp):
            cle = description.split(' : ')[0].strip()
            if cle not in seen:
                # Si nn déjà détecté, ne pas ajouter n
                if cle == 'n' and 'nn' in seen:
                    continue
                notes.append(description)
                seen.add(cle)
    return ' | '.join(notes)

df['notes_spectre'] = df['sp_type'].apply(decoder_sptype)

# ── Tableau HTML résultat ─────────────────────────────────────────────────────
def make_table(df):
    rows = ''
    for _, r in df.iterrows():
        bv   = f"{r['B_V']:.2f}" if pd.notna(r.get('B_V')) else '—'
        plx  = f"{r['parallax_mas']:.2f}" if 'parallax_mas' in r.index and pd.notna(r.get('parallax_mas')) else '—'
        dist = f"{1000/r['parallax_mas']:.0f}" if 'parallax_mas' in r.index and pd.notna(r.get('parallax_mas')) and r['parallax_mas'] > 0.5 else '—'
        ra_s = f"{r['ra']:.4f}"  if pd.notna(r.get('ra'))  else '—'
        de_s = f"{r['dec']:+.4f}" if pd.notna(r.get('dec')) else '—'
        notes = r.get('notes_spectre', '')
        notes_html = f'<span style="color:#444;font-size:0.82em">{notes}</span>' if notes else ''
        rows += (
            f'<tr>'
            f'<td><b>{r["main_id"]}</b></td>'
            f'<td style="text-align:center">{r["mag_V"]:.2f}</td>'
            f'<td style="text-align:center">{bv}</td>'
            f'<td><code>{r["sp_type"]}</code></td>'
            f'<td>{r.get("lum_label","—")}</td>'
            f'<td style="text-align:right">{ra_s}</td>'
            f'<td style="text-align:right">{de_s}</td>'
            f'<td style="text-align:center">{plx}</td>'
            f'<td style="text-align:center">{dist}</td>'
            f'<td>{notes_html}</td>'
            f'</tr>\n'
        )
    filtre_str  = ', '.join(FILTRE_SPECTRAL)  if FILTRE_SPECTRAL  else 'tous types'
    classes_str = ', '.join(CLASSES_LUMINOSITE) if CLASSES_LUMINOSITE else 'toutes classes'
    return (
        '<style>'
        '.sp-table{border-collapse:collapse;font-family:monospace;font-size:.87em;width:100%}'
        '.sp-table th{background:#1a2744;color:#eee;padding:5px 9px;text-align:left;white-space:nowrap}'
        '.sp-table td{border-bottom:1px solid #ddd;padding:3px 8px;vertical-align:top}'
        '.sp-table tr:hover{background:#f0f4ff}'
        '</style>'
        f'<h3>🔭 {CONSTELLATION} | V {MAG_MIN}–{MAG_MAX} | {filtre_str} | {classes_str} | {len(df)} étoiles</h3>'
        '<table class="sp-table"><tr>'
        '<th>Nom</th><th>V</th><th>B−V</th><th>Type sp.</th><th>Classe</th>'
        '<th>RA (°)</th><th>Dec (°)</th><th>π (mas)</th><th>d (pc)</th><th>Notes spectre</th>'
        f'</tr>\n{rows}</table>'
    )

if len(df) > 0:
    display(HTML(make_table(df)))
else:
    print('Aucune cible à afficher.')


Nom,V,B−V,Type sp.,Classe,RA (°),Dec (°),π (mas),d (pc),Notes spectre
* eps Her,3.92,-0.01,A0V,Naine,255.0724,+30.9264,19.76,51,
* ome Her,4.58,-0.01,A2VpCrSr,Naine,246.3540,+14.0333,13.10,76,
* e Her,4.65,0.05,A2V,Naine,259.4177,+37.2915,18.65,54,
* pi. Ser,4.82,0.07,A3V,Naine,240.5737,+22.8045,18.70,53,
* 60 Her,4.87,0.13,A3V,Naine,256.3445,+12.7408,23.68,42,
* 70 Her,5.12,-0.03,A2V,Naine,260.2259,+24.4994,7.42,135,
* rho Her B,5.40,-0.00,A0Vn,Naine,260.9197,+37.1468,8.91,112,n : rotation rapide — raies élargies par v·sin i élevé
* 25 Her,5.54,0.17,A5V,Naine,246.3507,+37.3941,13.19,76,
HD 161833,5.61,0.03,A1V,Naine,266.7835,+17.6970,7.64,131,
* 78 Her,5.65,0.00,A1V,Naine,262.9566,+28.4075,11.99,83,


---
## ⑤ Export csv

In [12]:
# ── Export CSV ───────────────────────────────────────────────────

filename = f'cibles_spectro_{CONSTELLATION}_V{MAG_MIN:.1f}-{MAG_MAX:.1f}.csv'
df.to_csv(filename, index=False, sep=';', decimal=',', encoding='utf-8-sig')

# Téléchargement automatique si Colab
try:
    from google.colab import files
    files.download(filename)
    print(f'✅ Fichier téléchargé : {filename}')
except ImportError:
    print(f'✅ Fichier sauvegardé : {filename} (téléchargement non disponible hors Colab)')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Fichier téléchargé : cibles_spectro_Her_V2.0-8.0.csv


---
## Aide-mémoire

### Types spectraux et raies caractéristiques

| Type | T_eff | Raies dominantes |
|---|---|---|
| O | >30 000 K | He II, He I, H (faibles) |
| B | 10 000–30 000 K | He I, H de Balmer |
| A | 7 500–10 000 K | H de Balmer très intenses |
| F | 6 000–7 500 K | H modérées, Ca II K&H, métaux |
| G | 5 200–6 000 K | Ca II K&H, G-band CH, Fe |
| K | 3 700–5 200 K | Ca II, bandes TiO naissantes, Fe |
| M | <3 700 K | Bandes TiO dominantes, CaH |
